# Tái lập *Cybersecurity Risk* (RFS 2023) — báo cáo tiến độ và kiểm chứng

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khoaminh2957/cyberrisk-replication/blob/main/cyberrisk_replication.ipynb)

Bài gốc: Florackis, C., Louca, C., Michaely, R. & Weber, M. (2023). *Cybersecurity Risk.*
**The Review of Financial Studies** 36(1), 351–407.

Bài đo rủi ro an ninh mạng của từng công ty niêm yết ở Mỹ từ mục *Item 1A Risk Factors* của báo cáo 10-K:
điểm của một công ty là độ giống trung bình (cosine) giữa đoạn viết về an ninh mạng của nó và của các
công ty vừa bị tấn công trong năm trước. Sau đó bài kiểm xem điểm này có được thị trường định giá không.

Notebook này làm hai việc:

1. **Tóm tắt đã làm được gì, chưa làm được gì và vì sao** (mục 1–2).
2. **Chạy lại để tự kiểm chứng** (mục 3–7): đối chiếu với ground truth của chính bài, dựng lại bảng nối
   tên, rồi dựng lại mọi con số trong `EVALUATION.md` mục 9 từ kết quả đã lưu.

Toàn bộ chạy được trên Colab miễn phí. Các ô nặng (tải 8,390 báo cáo 10-K) là tuỳ chọn và có cảnh báo thời gian.

In [ ]:
# --- Cài đặt (Colab: clone repo; máy cá nhân: bỏ qua bước clone) ---------------------------
import os, sys, subprocess

REPO = "https://github.com/khoaminh2957/cyberrisk-replication.git"
if not os.path.exists("cyberrisk"):                      # chưa ở trong repo
    if not os.path.exists("cyberrisk-replication"):      # chưa clone lần nào
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    os.chdir("cyberrisk-replication")
sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# SEC fair access: đổi thành tên và email của bạn trước khi tải bất kỳ thứ gì từ EDGAR
os.environ["EDGAR_USER_AGENT"] = "academic replication contact@example.edu"

import pandas as pd, numpy as np, json
pd.set_option("display.width", 200)
DATA = os.path.join(os.getcwd(), "cyberrisk", "data")
summary = json.load(open(f"{DATA}/results/summary.json"))
print("lần chạy:", summary["run"], "| hồ sơ 10-K:", summary["n_filings"], "| công ty-năm chấm điểm:", summary["n_scored"])

## 1. Đã làm được

| Phần của bài | Trạng thái | Bằng chứng |
|---|---|---|
| Phụ lục A: bảng từ khóa + thuật toán trích câu | Xong | trùng **68/68** quyết định bắt/bỏ câu trên 4 hồ sơ 10-K thật mà bài in ở Phụ lục A.2 (mục 3 bên dưới) |
| §2.2 tải 10-K, tách Item 1A, loại "incorporate by reference" | Xong | 8,390 hồ sơ 2005–2019, không còn lỗi |
| §2.3 mẫu huấn luyện từ dữ liệu PRC | Xong | 288 vụ tấn công nối với 215 công ty, theo quy tắc cố định trong `link_prc.py` (mục 4) |
| §2.4 loại từ, gốc từ, tần suất ≥ 10, phương trình (1)–(2) | Xong | tính lại độc lập bằng numpy trên 8 công ty-năm: trùng tới 1e-9 |
| §3 Bảng 1, Bảng 2, Bảng 3 (phần văn bản), Hình 1, Hình 2 | Xong | mục 5 |
| §3.5 Bảng 6 Model 1 | Xong, thiết kế khác | case-control, vì không liệt kê được 41,140 công ty-năm của bài (mục 6) |
| Bảng 4–5, 7–12, IA7–IA14 | Code xong, **chưa ra số** | thiếu WRDS (mục 2) |

Kiểm chứng đã chạy: 74 test tự động (73 pass, 1 test mạng chạy riêng), kết quả giống nhau qua 5 hash seed,
hai vòng audit độc lập tìm ra 21 khẳng định sai trong tài liệu của chính tôi — tất cả đã sửa và ghi lại
trong `cyberrisk/EVALUATION.md` mục 11.

## 2. Chưa làm được, và vì sao

Nửa tài chính bị chặn bởi **dữ liệu có giấy phép**, không phải bởi code: mọi hàm đã viết theo bài, có test
trên dữ liệu tổng hợp và ví dụ tính tay, nhưng chưa hàm nào chạy trên dữ liệu thật.

| Thiếu | Chặn phần nào |
|---|---|
| CRSP (giá và lợi suất, gồm cả công ty đã hủy niêm yết) | Bảng 5, 7–12 và toàn bộ Internet Appendix |
| Compustat | 7/16 biến của Bảng 3 (cùng 13F và BoardEx là 10/16); biến kiểm soát của Bảng 4–6 |
| Thomson-Reuters 13F | sở hữu tổ chức |
| BoardEx | thành viên độc lập, ủy ban rủi ro |
| CRSP–Compustat link + WRDS SEC Analytics | cầu nối CIK + năm tài chính → gvkey → permno |
| Factiva | cờ "vụ lớn"; đang dùng biến thể "mọi vụ" mà bài báo cáo là *unchanged* |
| FactSet Revere + Bloomberg | Bảng 12 (khách hàng SolarWinds, chỉ số chú ý) |

**Hệ quả lớn nhất: kết quả chính của bài — danh mục điểm cao sinh lời hơn tới 8.3%/năm — chưa kiểm được dòng nào.**

Thiếu WRDS còn làm nửa văn bản phải dùng proxy, và chỗ nào cũng đo được ảnh hưởng:

* **Vũ trụ công ty**: bài dùng CRSP; ở đây đọc tên sàn trong 40,000 ký tự đầu của 10-K. Tỷ lệ điểm 0 năm 2018
  là 9.2% cho nhóm "niêm yết" và 50.0% cho nhóm còn lại, nên bộ lọc này quyết định phân phối điểm.
* **Mẫu**: không liệt kê được 41,140 công ty-năm của bài nên phải rút ngẫu nhiên; từ vựng còn 2,092 gốc từ so với 3,210.
* **Bảng 6 Model 1**: phải dùng case-control; hệ số 1.320 so với 0.961 của bài.
* **Ngành**: SIC hiện tại của SEC thay cho SIC theo từng năm của Compustat.

## 3. Kiểm chứng 1 — ground truth của chính bài (Phụ lục A.2)

Phụ lục A.2 của bài in **từng câu** mà thuật toán của họ bắt hoặc bỏ trên 10-K năm tài chính 2017 của
Apple, Abbott, General Motors và Verizon. Đây là phép thử mạnh nhất có được: tải đúng 4 hồ sơ đó từ EDGAR,
chạy thuật toán ở đây, so từng câu.

Ô dưới tải khoảng 9 MB từ SEC. **Đổi `EDGAR_USER_AGENT` ở ô đầu thành tên và email của bạn trước khi chạy.**

In [ ]:
RUN_NETWORK = True   # đặt False nếu không muốn gọi mạng

if RUN_NETWORK:
    subprocess.run([sys.executable, "-m", "cyberrisk.fetch_data", "appendix-a2"], check=True)
    subprocess.run([sys.executable, "-m", "pytest", "cyberrisk/tests/test_appendix_a2.py", "-q", "-s"], check=False)
else:
    print("bỏ qua (RUN_NETWORK = False)")

Mỗi công ty phải bắt đúng số câu bài liệt kê là đã bắt (Apple 23, Abbott 8, GM 20, Verizon 13 — gồm cả
các câu ngoài đoạn chính) và phải bỏ đúng 4 câu mà bài nói thuật toán bỏ sót. Tỷ lệ bài tự in cho riêng
đoạn liên quan là 19/19, 8/8, 18/20, 13/15. Nhãn *loại câu* trùng 58/64 — 58 là mức tối đa với mọi thứ tự ưu tiên (đã thử cả
24 hoán vị), vì 6 câu còn lại bài gắn nhãn không suy ra được từ bảng luật đã in. Nhãn không ảnh hưởng tới
thước đo, vì vector lấy toàn bộ câu bắt được.

## 4. Kiểm chứng 2 — bảng nối tên PRC → công ty nộp 10-K

Bài nối tên công ty trong dữ liệu Privacy Rights Clearinghouse với CRSP/Compustat **bằng tay** và không công bố
bảng nối. Ở đây bảng nối được dựng lại bằng một quy tắc cố định, ghi trong docstring của `link_prc.py`:
khớp chính xác tên đã chuẩn hoá, cộng các quyết định tay theo đúng một luật (nhận chính công ty dưới mọi tên đã
dùng, hoặc đơn vị mang token đặc trưng của tên công ty; từ chối đơn vị khác tên, công ty tư nhân, 20-F, tên mơ hồ).

Ô dưới tải lại bản PRC công khai (khoảng 5 MB) và dựng lại bảng nối từ đầu, rồi so từng dòng với bản đã lưu trong repo.

In [ ]:
if RUN_NETWORK:
    import requests, tempfile
    from cyberrisk import link_prc
    PRC_MIRROR = "https://raw.githubusercontent.com/jbukuts/databreaches/HEAD/data/data_breaches.csv"
    src = f"{DATA}/prc/prc_export_jbukuts.csv"
    if not os.path.exists(src):
        open(src, "wb").write(requests.get(PRC_MIRROR, timeout=120).content)
    tmp = tempfile.mktemp(suffix=".csv")
    out = link_prc.build(src, f"{DATA}/edgar_10k_filers_2005_2019.json", tmp)
    kw = dict(dtype={"cik": str, "prc_id": str})
    got, ref = pd.read_csv(tmp, **kw), pd.read_csv(f"{DATA}/prc/link_prc_cik.csv", **kw)
    print("số dòng dựng lại:", len(got), "| trùng từng dòng với bản đã lưu:", got.equals(ref))
    print("cách nối:", out["method"].value_counts().to_dict(), "| số công ty:", got["cik"].nunique())
else:
    ref = pd.read_csv(f"{DATA}/prc/link_prc_cik.csv", dtype={"cik": str})
    print("bản đã lưu:", len(ref), "vụ,", ref["cik"].nunique(), "công ty")

## 5. Dựng lại các con số của `EVALUATION.md` mục 9

`cyberrisk/data/results/scores_v6.csv` là kết quả của lần chạy đầy đủ: 7,198 hồ sơ có Item 1A, mỗi dòng là một
báo cáo 10-K với điểm rủi ro và các biến ngôn ngữ. Từ file này, mọi bảng của nửa văn bản dựng lại được trong vài giây
bằng đúng code đã dùng để tạo ra chúng — không cần tải lại EDGAR.

In [ ]:
from cyberrisk import replicate_text as R

ok = pd.read_csv(f"{DATA}/results/scores_v6.csv", dtype={"cik": str, "sic": str}, parse_dates=["filing_date"])
link = pd.read_csv(f"{DATA}/prc/link_prc_cik.csv", dtype={"cik": str}, parse_dates=["attack_date"])
rep = R.report(ok, [], {}, link)

print("mẫu chấm điểm:", len(R.scored_sample(ok)), "công ty-năm\n")
print("Bảng 3 — phân phối điểm rủi ro"); print(rep["table3"].round(3).to_string())

if "readability" in rep:                       # kích thước file nộp, lấy từ kho bulk của SEC
    print(f"\nBảng 3 — Readability (byte), N = {rep['readability_n']}")
    print(rep["readability"].apply(lambda c: c.map("{:,.0f}".format)).to_string())
    print("\ndạng log:"); print(rep["readability_ln"].round(2).to_string())

In [ ]:
print("Bảng 1 — mười công ty-năm bài in điểm")
t1 = rep["table1"].copy(); t1["chênh"] = (t1["ours"] - t1["paper"]).round(3)
print(t1.round(3).to_string(index=False))
print("\nsai lệch tuyệt đối trung bình:", round((t1['ours'] - t1['paper']).abs().mean(), 3))

In [ ]:
print("Bảng 2 — tương quan của điểm với đặc điểm ngôn ngữ (công ty có đoạn viết)")
print(rep["table2"].round(3).to_string())
print("\nHình 2 — trung bình theo ngành Fama-French 12")
print(rep["figure2"].round(3).to_string())
print("tương quan hạng Spearman với thứ tự của bài:", round(rep["figure2_spearman"], 3))

In [ ]:
print("Mẫu huấn luyện — số vụ tấn công theo năm (bài: 175)")
print(rep["attack_counts"].to_string())

In [ ]:
import matplotlib.pyplot as plt

by = rep["by_year"]
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.errorbar(by.index, by["mean"], yerr=1.96 * by["se"], fmt="o-", capsize=3, label="tái lập (KTC 95%)")
ax.plot(by.index, by["paper_mean(chart)"], "s--", color="grey", fillstyle="none", label="bài (đọc từ Hình 1)")
ax.set_xlabel("năm tài chính"); ax.set_ylabel("điểm rủi ro trung bình")
ax.set_title("Hình 1: điểm trung bình theo năm"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

print("sai lệch tuyệt đối trung bình so với bài:", round((by["mean"] - by["paper_mean(chart)"]).abs().mean(), 3))
print("số năm khoảng tin cậy 95% chứa số của bài:",
      int(((by["paper_mean(chart)"] >= by["mean"] - 1.96 * by["se"]) &
           (by["paper_mean(chart)"] <= by["mean"] + 1.96 * by["se"])).sum()), "/ 12")

## 5b. So với thước đo của chính tác giả

Tác giả công bố thước đo của họ trên Harvard Dataverse (DOI `10.7910/DVN/LCVVG5`, giấy phép CC0). Bộ đó
có code SAS và Stata cho phần tài chính, nhưng **không có code xử lý văn bản** — đúng phần notebook này
dựng lại. Bù lại, file `flmw_rfs.dta` chứa 44,972 công ty-năm kèm điểm của họ, nên so được từng công ty-năm.

Khóa của họ là gvkey (Compustat), khóa ở đây là CIK, nên cầu nối là tên công ty đã chuẩn hóa, chỉ giữ
những tên định danh duy nhất một gvkey và một CIK. Ô dưới tải file của họ (3.7 MB) rồi tính.

In [ ]:
if RUN_NETWORK:
    from cyberrisk import compare_authors
    compare_authors.main(["cyberrisk/data/results/scores_v6.csv"])
else:
    print("bỏ qua (RUN_NETWORK = False)")

## 6. Bảng 6 Model 1 — điểm có dự báo vụ tấn công năm sau không

Bài chạy logit trên 41,140 công ty-năm có đủ dữ liệu tài chính. Không có Compustat thì không liệt kê được mẫu đó,
nên ở đây dùng **case-control**: nhóm đối chứng là mẫu rút ngẫu nhiên, nhóm ca là công ty-năm ngay trước mỗi vụ
tấn công. Với logit, cách chọn mẫu phụ thuộc kết quả này vẫn cho hệ số độ dốc nhất quán (Prentice & Pyke, 1979);
chỉ hệ số chặn bị dịch, và hiệu ứng cố định năm hấp thụ phần đó.

In [ ]:
t6 = R.table6_model1(ok, link)
print(f"bản này: hệ số {t6['coef']:.3f}, t {t6['t']:.2f}, N {t6['n']:,}, số ca {t6['cases']}")
print(f"bài:     hệ số {t6['paper']['coef']:.3f}, t {t6['paper']['t']:.2f}, N {t6['paper']['n']:,}\n")

v = summary["table6_variants"]
print("năm biến thể chốt trước khi chạy, cộng bootstrap theo công ty:")
for k in ("V0_calendar_year", "V1_12m_after_fiscal_year_end", "V2_12m_after_filing",
          "V3_ff48_fixed_effects", "V4_previous_attack_control"):
    d = v[k]; print(f"  {k:32s} hệ số {d['coef']:.3f}  KTC 95% [{d['ci95'][0]:.3f}, {d['ci95'][1]:.3f}]  ca {d['cases']}")
b = v["firm_bootstrap_V0"]
print(f"  bootstrap V0 ({b['draws']} mẫu)          trung vị {b['median']:.3f}  KTC 95% {b['ci95']}  số mẫu ≤ 0.961: {b['n_le_paper_0.961']}")

Quan sát, không phải giải thích: khoảng tin cậy của V2, V3, V4 chứa 0.961; của V0 và V1 thì không.
Không biến thể nào đưa hệ số về gần 0.96. Ba ứng viên còn lại đều cần dữ liệu không có (toàn bộ công ty-năm
Compustat, cờ "major" của Factiva, bảng nối tên của tác giả), nên nguyên nhân khoảng cách này
vẫn là **MECHANISM: UNKNOWN**.

## 7. Bộ test

74 test: luật từ khóa và cửa sổ tìm kiếm, tách câu, gốc từ, phương trình (1)–(2), biến Phụ lục B đối chiếu
với ví dụ tính tay, các ước lượng kinh tế lượng trên dữ liệu tổng hợp có cài sẵn hiệu ứng, và các con số
chính của lần chạy được ghim lại để một thay đổi code làm lệch số là fail ngay.

Trong bản clone sạch: 66 pass và 10 skip, vì 10 test cần dữ liệu không kèm repo. Sau khi chạy hai ô tải ở
mục 3 và 4 ở trên: 71 pass và 5 skip. Trên máy có đủ dữ liệu: 75 pass và 1 test mạng.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "cyberrisk/tests", "-q"], check=False)

## 8. Chạy lại toàn bộ từ đầu (tuỳ chọn, 45–60 phút)

Ô dưới tải lại toàn bộ 8,390 báo cáo 10-K từ EDGAR rồi tính lại từ số không. Colab miễn phí chạy được, nhưng
cần khoảng 460 MB đĩa, và cần thêm hai file không nằm trong repo: từ điển Loughran–McDonald (biến ngôn ngữ
của Bảng 2) và bảng SIC theo CIK của SEC. Xem `cyberrisk/README.md` mục "Reproducing the numbers".

```python
# !python cyberrisk/data/run/run_download.py        # ~45-60 phút, 8 tiến trình, ~2 request/giây
# from cyberrisk import replicate_text as R
# ok, vocab, counts, link, df = R.compute("cyberrisk/data/run/v6", ...)
```

## 9. Việc còn mở

* 6 lỗi code đã xác nhận ở nửa tài chính chưa sửa (`EVALUATION.md` mục 11.3) — chưa ảnh hưởng con số nào,
  vì nửa đó chưa chạy trên dữ liệu thật.
* `readability` (dung lượng file nộp đầy đủ) chưa có trong lần chạy này: bộ tải dừng ở cuối văn bản 10-K.
* Độ phủ đọc tay khi nối tên: 334/698 cặp ứng viên được đọc ở lượt đầu, phần còn lại chỉ được xét qua hai
  lượt đọc theo token hiếm và theo chữ đầu.
* Mọi chỗ lệch với bài đều ghi **MECHANISM: UNKNOWN** kèm danh sách ứng viên; không ứng viên nào được chọn
  khi chưa có thí nghiệm phân biệt.

Tài liệu đầy đủ: [`cyberrisk/EVALUATION.md`](cyberrisk/EVALUATION.md) (mục 9 = số liệu, mục 10 = 11 lỗi đã
tìm và sửa, mục 11 = vòng kiểm tra hallucination). Bản đồ bài → code: [`cyberrisk/README.md`](cyberrisk/README.md).

Nguồn dữ liệu công khai: SEC EDGAR, Privacy Rights Clearinghouse, thư viện Kenneth French, từ điển
Loughran–McDonald, WordNet.